<a href="https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fatima-05/FlyRank-ML/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Build a short ranked queue a human can trust:
- score from the model
- plain action label
- reason codes in ordinary words

In [7]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

if not Path("data/raw/content_refresh_anonymized.csv").exists():
    if not Path("/content/FlyRank-ML").exists():
        !git clone https://github.com/Fatima-05/FlyRank-ML.git /content/FlyRank-ML
    os.chdir("/content/FlyRank-ML")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

feature_cols = [c for c in [
    "impressions_90d", "clicks_90d", "ctr", "content_age_days",
    "word_count", "avg_position", "search_volume"
] if c in df.columns]

X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
y = df["is_declining"]
groups = df["client_id"] if "client_id" in df.columns else pd.Series(np.arange(len(df)))

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

scored = df.iloc[test_idx].copy()
scored["model_score"] = rf.predict_proba(X.iloc[test_idx])[:, 1]

def reason_codes(row):
    reasons = []
    if (row.get("impressions_90d", 0) or 0) >= 2000:
        reasons.append("high_impressions")
    if (row.get("ctr", 1) or 1) < 0.02:
        reasons.append("low_ctr")
    if (row.get("content_age_days", 0) or 0) >= 180:
        reasons.append("aging")
    pos = row.get("avg_position", None)
    if pos is not None and 4 <= pos <= 15:
        reasons.append("striking_distance")
    if (row.get("word_count", 0) or 0) > 0 and (row.get("word_count", 0) or 0) < 400:
        reasons.append("thin_content")
    return ",".join(reasons) if reasons else "review"

def action_label(score, median_score):
    if score >= median_score:
        return "refresh"
    return "monitor"

med = scored["model_score"].median()
scored["reason_code"] = scored.apply(reason_codes, axis=1)
scored["action"] = scored["model_score"].apply(lambda s: action_label(s, med))

queue = scored.sort_values("model_score", ascending=False).head(20).copy()
queue.insert(0, "rank", range(1, len(queue) + 1))

show_cols = [c for c in [
    "rank", "content_id", "model_score", "action", "reason_code",
    "impressions_90d", "ctr", "avg_position", "content_age_days"
] if c in queue.columns]

queue[show_cols]

,rank,content_id,model_score,action,reason_code,impressions_90d,ctr,avg_position,content_age_days
12651,1,content_2a228ce7aa1b,0.827218,refresh,striking_distance,155,0.00,7.5,174
21729,2,content_7766ffacdcfa,0.823919,refresh,"aging,striking_distance",657,0.00,9.3,275
6228,3,content_e988c1699454,0.823176,refresh,"high_impressions,aging",2197,0.00,21.5,275
12591,4,content_7e24f3b27d73,0.822540,refresh,"aging,striking_distance",825,0.00,11.6,275
15705,5,content_4dd569ee33c9,0.820898,refresh,"aging,striking_distance",361,0.00,14.8,275
18475,6,content_197a5b1ed096,0.820840,refresh,"aging,striking_distance",605,0.00,14.4,223
22042,7,content_2ba626fea4d6,0.820807,refresh,"aging,striking_distance",360,0.00,7.2,275
22730,8,content_bb1edbf0ba6c,0.820412,refresh,"aging,striking_distance",1290,0.00,12.4,280
28718,9,content_ef6e7d7cfe15,0.818753,refresh,aging,264,0.00,22.2,271
20736,10,content_41baf0722ad9,0.818490,refresh,"high_impressions,aging,striking_distance",3115,0.00,12.8,275


## 2. Intended use and limits

Who: a content editor or SEO reviewer on a small team.

What for: decide which pages to open first in this week’s refresh pass.

Where it stops being valid:
- not auto-publishing or auto-rewriting
- not a promise of traffic lift after refresh
- not valid if the underlying tracking or content inventory changes heavily and the model is not re-scored
- not a substitute for legal, brand, or factual review of the page itself

In [8]:
print("Intended use: human-facing ranked refresh queue")
print("Stops at: prioritisation for review — not auto-publish, not causal impact claims")

Intended use: human-facing ranked refresh queue
Stops at: prioritisation for review — not auto-publish, not causal impact claims


## 3. Human review + the no-go list

Before acting, a person should check:
- Is the page still live and still relevant to the business?
- Is low CTR expected for this intent (brand/navigational)?
- Is the real issue technical, cannibalization, or a SERP feature — not content age?
- Has this page already been refreshed offline recently?

Never automate:
- bulk publish of rewritten pages from the queue alone
- deletion/merge without human confirmation
- any action on pages that need legal or compliance review
- trusting the score when reason codes are empty or contradictory to what the page shows

In [9]:
nogo = pd.DataFrame({
    "must_check": [
        "page still relevant",
        "CTR may be normal for intent",
        "technical or cannibalization causes",
        "recent offline refresh already done",
    ],
    "never_automate": [
        "auto-publish rewrites",
        "auto-delete/merge",
        "compliance-sensitive edits",
        "score-only actions with no page review",
    ],
})
nogo

,must_check,never_automate
0,page still relevant,auto-publish rewrites
1,CTR may be normal for intent,auto-delete/merge
2,technical or cannibalization causes,compliance-sensitive edits
3,recent offline refresh already done,score-only actions with no page review


## 4. Monitoring / retrain triggers

Recommendations go stale when inputs drift or the queue stops helping.

Retrain / re-score triggers:
- monthly (or sprint) re-pull of page signals
- large change in site template, tracking, or content inventory
- top-queue precision on a fresh labeled sample falls toward base rate
- editors repeatedly dismiss the same reason codes as unhelpful

In [10]:
triggers = pd.DataFrame({
    "trigger": [
        "scheduled monthly rescore",
        "tracking or template change",
        "top-queue precision near base rate",
        "repeated editor dismissals of same reasons",
    ],
    "response": [
        "rebuild features + scores",
        "rebuild features + check leakage",
        "audit labels and features",
        "revise reason codes / thresholds",
    ],
})
triggers

,trigger,response
0,scheduled monthly rescore,rebuild features + scores
1,tracking or template change,rebuild features + check leakage
2,top-queue precision near base rate,audit labels and features
3,repeated editor dismissals of same reasons,revise reason codes / thresholds


## 5. Exports for the paper

Write a public-safe queue sample to work/outputs/ for the paper and report.

In [11]:
from pathlib import Path

out = Path("work/outputs")
out.mkdir(parents=True, exist_ok=True)

export_cols = [c for c in [
    "rank", "content_id", "model_score", "action", "reason_code",
    "impressions_90d", "ctr", "avg_position", "content_age_days"
] if c in queue.columns]

export_path = out / "action_playbook_top20.csv"
queue[export_cols].to_csv(export_path, index=False)
print("Wrote", export_path)
queue[export_cols].head(10)

Wrote work/outputs/action_playbook_top20.csv


,rank,content_id,model_score,action,reason_code,impressions_90d,ctr,avg_position,content_age_days
12651,1,content_2a228ce7aa1b,0.827218,refresh,striking_distance,155,0.0,7.5,174
21729,2,content_7766ffacdcfa,0.823919,refresh,"aging,striking_distance",657,0.0,9.3,275
6228,3,content_e988c1699454,0.823176,refresh,"high_impressions,aging",2197,0.0,21.5,275
12591,4,content_7e24f3b27d73,0.822540,refresh,"aging,striking_distance",825,0.0,11.6,275
15705,5,content_4dd569ee33c9,0.820898,refresh,"aging,striking_distance",361,0.0,14.8,275
18475,6,content_197a5b1ed096,0.820840,refresh,"aging,striking_distance",605,0.0,14.4,223
22042,7,content_2ba626fea4d6,0.820807,refresh,"aging,striking_distance",360,0.0,7.2,275
22730,8,content_bb1edbf0ba6c,0.820412,refresh,"aging,striking_distance",1290,0.0,12.4,280
28718,9,content_ef6e7d7cfe15,0.818753,refresh,aging,264,0.0,22.2,271
20736,10,content_41baf0722ad9,0.818490,refresh,"high_impressions,aging,striking_distance",3115,0.0,12.8,275


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.